In [2]:
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


In [4]:
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])

5 (2, 3)
1 8
99
2 0.5
(2, 3) 0.25


Мои предсказания и объяснения:

5, (2, 3) — a[1, 2] берёт элемент второй строки и третьего столбца; shape показывает размер массива.
1, 8 — argmax() возвращает индекс первого максимума, а max() само максимальное значение.
99 — срез a[::3] обычно является представлением исходного массива, поэтому изменение b[0] меняет a[0].
2, 0.5 — больше 3 два элемента (5 и 7), среднее булевой маски равно доле истинных значений.
(2, 3), 0.25 — zeros создаёт массив формы 2×3, а второй элемент linspace из пяти точек от 0 до 1 равен 0.25.

[База]
1. Найдите позицию (argmax) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску paid и ~paid.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [5]:
i_max = amount.argmax()
print("Позиция:", i_max)
print("Сумма:", amount[i_max], "₸")
print("Клиент:", cust[i_max])
print("Город:", city[i_max])

print("Выручка оплаченных:", amount[paid].sum(), "₸")
print("Сумма неоплаченных:", amount[~paid].sum(), "₸")

shymkent = city == "Шымкент"
print("Заказов в Шымкенте:", shymkent.sum())
print("Доля:", round(shymkent.mean() * 100, 2), "%")


Позиция: 1001
Сумма: 9999990.0 ₸
Клиент: TEST
Город: Алматы
Выручка оплаченных: 101909360.0 ₸
Сумма неоплаченных: 7974000.0 ₸
Заказов в Шымкенте: 804
Доля: 14.72 %


[Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для каждого города — доля заказов, доставленных за 1–2 дня. Список городов: np.unique(city); цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:

In [6]:
discount_mask = paid & (amount > 50000)
loss_all = amount[discount_mask].sum() * 0.15

discount_without_test = discount_mask & (cust != "TEST")
loss_without_test = amount[discount_without_test].sum() * 0.15

print("Недополученная выручка со всеми заказами:", round(loss_all, 2), "₸")
print("Недополученная выручка без TEST:", round(loss_without_test, 2), "₸")
print("Разница:", round(loss_all - loss_without_test, 2), "₸")
print("Объяснение: заказы клиента TEST сильно влияют на сумму скидки, потому что среди них есть крупные заказы.")

for c in np.unique(city):
    city_mask = city == c
    valid_days = city_mask & ~np.isnan(days)
    fast_delivery = valid_days & (days >= 1) & (days <= 2)
    share = fast_delivery.sum() / valid_days.sum() if valid_days.sum() > 0 else np.nan
    print(c, "—", round(share * 100, 2), "%")

avg_check = amount[paid].mean()
print("Средний чек оплаченного заказа:", round(avg_check, 2), "₸")


Недополученная выручка со всеми заказами: 5702422.5 ₸
Недополученная выручка без TEST: 1202427.0 ₸
Разница: 4499995.5 ₸
Объяснение: заказы клиента TEST сильно влияют на сумму скидки, потому что среди них есть крупные заказы.
Актобе — 62.85 %
Алматы — 66.51 %
Астана — 65.41 %
Атырау — 66.67 %
Караганда — 71.01 %
Шымкент — 30.69 %
Средний чек оплаченного заказа: 20780.86 ₸
